In [ ]:
pip install numpy pandas pycountry pyarrow plotly scikit-learn streamlit duckdb scipy statsmodels Pillow openpyxl xlsxwriter xlrd matplotlib

# ETL TFM SIT 2026

In [1]:
import pandas as pd
import numpy as np
import glob
import pycountry
import gettext
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
import gc
import warnings
import pyarrow as pa
import pyarrow.parquet as pq
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')

In [2]:
# ---------------------------------------------------------------------------
# Estilo visual global — coherente con estetica academica
# ---------------------------------------------------------------------------
plt.rcParams.update({
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : '#F7F7F7',
    'axes.edgecolor'    : '#CCCCCC',
    'axes.grid'         : True,
    'grid.color'        : 'white',
    'grid.linewidth'    : 1.0,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'font.family'       : 'serif',
    'font.size'         : 10,
    'axes.titlesize'    : 12,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 10,
    'xtick.labelsize'   : 9,
    'ytick.labelsize'   : 9,
    'legend.fontsize'   : 9,
    'figure.dpi'        : 150,
})

AZUL_TFM    = '#1B4F8A'
GRIS_TFM    = '#5A6472'
ROJO_TFM    = '#C0392B'
VERDE_TFM   = '#1E7A4D'
NARANJA_TFM = '#D4680A'


def guardar(fig, nombre, fig_path):
    ruta = fig_path / nombre
    fig.savefig(ruta, bbox_inches='tight', dpi=150)
    plt.close(fig)
    print(f"  [figura guardada] {ruta.name}")


def tabla_resumen(df_tabla, titulo, col_width=72):
    linea = "-" * col_width
    print(f"\n{linea}")
    print(f"  {titulo}")
    print(linea)
    print(df_tabla.to_string())
    print(linea)

In [3]:
# =============================================================================
# SESION 1 — Primer contacto: que hay en la carpeta raw?
# =============================================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# -----------------------------------------------------------------------------
# Detectar automaticamente la raiz del proyecto TFM_SIT2026
# Funciona desde notebooks/, scripts o cualquier ordenador
# -----------------------------------------------------------------------------

def encontrar_raiz_proyecto(nombre_raiz="TFM_SIT2026"):
    ruta_actual = Path.cwd().resolve()

    for ruta in [ruta_actual] + list(ruta_actual.parents):
        if ruta.name == nombre_raiz:
            return ruta

    raise FileNotFoundError(
        f"No se encontro la carpeta raiz '{nombre_raiz}' "
        f"en la ruta actual ni en directorios superiores."
    )

PATH_BASE = encontrar_raiz_proyecto()

# -----------------------------------------------------------------------------
# Estructura estandar del proyecto
# -----------------------------------------------------------------------------

raw_path       = PATH_BASE / "data" / "raw"
processed_path = PATH_BASE / "data" / "processed"
fig_path       = PATH_BASE / "outputs" / "figures"
notebooks_path = PATH_BASE / "notebooks"
assets_path    = PATH_BASE / "assets"

# -----------------------------------------------------------------------------
# Crear carpetas necesarias si no existen
# -----------------------------------------------------------------------------

raw_path.mkdir(parents=True, exist_ok=True)
processed_path.mkdir(parents=True, exist_ok=True)
fig_path.mkdir(parents=True, exist_ok=True)

print(f"Carpeta raiz detectada : {PATH_BASE}")
print(f"Directorio raw         : {raw_path}")
print(f"Existe                 : {raw_path.exists()}")

# -----------------------------------------------------------------------------
# Inventario de archivos
# -----------------------------------------------------------------------------

todos_los_archivos = sorted(raw_path.iterdir())

print(f"\nContenido del directorio raw ({len(todos_los_archivos)} elementos):")

for f in todos_los_archivos:
    if f.is_file():
        print(f"  {f.name:55}  {f.stat().st_size / 1e6:.2f} MB")
    else:
        print(f"  {f.name:55}  <DIR>")

# -----------------------------------------------------------------------------
# FIGURA 1 — Inventario de archivos
# -----------------------------------------------------------------------------

info_archivos = pd.DataFrame([
    {
        'archivo': f.name,
        'MB': f.stat().st_size / 1e6
    }
    for f in todos_los_archivos
    if f.is_file()
]).sort_values('MB', ascending=True)

def color_por_extension(nombre):
    if '.tsv' in nombre:
        return AZUL_TFM
    if '.csv' in nombre:
        return GRIS_TFM
    return NARANJA_TFM

colores_inv = [color_por_extension(n) for n in info_archivos['archivo']]

fig, ax = plt.subplots(
    figsize=(10, max(4, len(info_archivos) * 0.32))
)

bars = ax.barh(
    info_archivos['archivo'],
    info_archivos['MB'],
    color=colores_inv,
    edgecolor='white',
    height=0.65
)

for bar, mb in zip(bars, info_archivos['MB']):
    ax.text(
        bar.get_width() + 0.05,
        bar.get_y() + bar.get_height() / 2,
        f'{mb:.1f} MB',
        va='center',
        fontsize=8,
        color=GRIS_TFM
    )

ax.set_xlabel('Tamano (MB)')
ax.set_title('Fig. 1 — Inventario de archivos fuente (directorio raw)')

legend_handles = [
    mpatches.Patch(color=AZUL_TFM,    label='.tsv  (COMEXT)'),
    mpatches.Patch(color=GRIS_TFM,    label='.csv  (macro / puertos)'),
    mpatches.Patch(color=NARANJA_TFM, label='.xls/x (distancias / brent)')
]

ax.legend(
    handles=legend_handles,
    loc='lower right',
    framealpha=0.9
)

plt.tight_layout()

guardar(fig, 'fig01_inventario_archivos.png', fig_path)

Carpeta raiz detectada : /Users/hugoa/Library/CloudStorage/OneDrive-UniversitatdeBarcelona/UB/Tesina/TFM_SIT2026
Directorio raw         : /Users/hugoa/Library/CloudStorage/OneDrive-UniversitatdeBarcelona/UB/Tesina/TFM_SIT2026/data/raw
Existe                 : True

Contenido del directorio raw (22 elementos):
  .DS_Store                                                0.01 MB
  CLVMNACSCAB1GQEA19.csv                                   0.00 MB
  DEXUSEU.csv                                              0.13 MB
  IR3TIB01EZM156N.csv                                      0.01 MB
  POILBREUSDM.csv                                          0.01 MB
  Ports.csv                                                0.52 MB
  brent_price.xls                                          0.05 MB
  dist_cepii.xls                                           8.14 MB
  ecb_rate.csv                                             0.15 MB
  eur_usd.csv                                              1.88 MB
  eurofxref-hist.cs

In [4]:
# =============================================================================
# SESION 2 — Abrir uno de los TSV y ver que tiene dentro
# =============================================================================

tsv_files = sorted(raw_path.glob("import_*.tsv"))
print(f"\nFicheros TSV encontrados: {len(tsv_files)}")
for f in tsv_files[:8]:
    print(f"  {f.name}")
if len(tsv_files) > 8:
    print(f"  ... y {len(tsv_files) - 8} mas")

primer_tsv = tsv_files[0]
print(f"\nAbriendo: {primer_tsv.name}")
df_vistazo = pd.read_csv(primer_tsv, sep='\t', nrows=10)
print(f"Columnas: {list(df_vistazo.columns)}")
print(df_vistazo.to_string())


Ficheros TSV encontrados: 8
  import_2002_2007_kg.tsv
  import_2002_2007_val.tsv
  import_2008_2013_kg.tsv
  import_2008_2013_val.tsv
  import_2014_2019_kg.tsv
  import_2014_2019_val.tsv
  import_2020_2025_kg.tsv
  import_2020_2025_val.tsv

Abriendo: import_2002_2007_kg.tsv
Columnas: ['REPORTER', 'PARTNER', 'PRODUCT', 'FLOW', 'TRANSPORT_MODE', 'PERIOD', 'INDICATORS', 'INDICATOR_VALUE']
  REPORTER PARTNER  PRODUCT  FLOW  TRANSPORT_MODE  PERIOD   INDICATORS  INDICATOR_VALUE
0       AT      AE        8     1               1  200510  QUANTITY_KG             1000
1       AT      AE       21     1               1  200711  QUANTITY_KG                0
2       AT      AE       25     1               1  200301  QUANTITY_KG            20800
3       AT      AE       25     1               1  200512  QUANTITY_KG            14400
4       AT      AE       25     1               1  200605  QUANTITY_KG            12000
5       AT      AE       26     1               1  200610  QUANTITY_KG           1

In [5]:
# =============================================================================
# SESION 3 — Estructura kg / val: confirmar claves de merge
# =============================================================================

kg_files  = sorted(raw_path.glob("import_*_kg.tsv"))
val_files = sorted(raw_path.glob("import_*_val.tsv"))
print(f"\nFicheros _kg.tsv : {len(kg_files)}")
print(f"Ficheros _val.tsv: {len(val_files)}")

kf = str(kg_files[0])
vf = kf.replace("_kg.tsv", "_val.tsv")

df_k_test = pd.read_csv(kf, sep='\t',
                         dtype={'PRODUCT': str, 'PARTNER': str, 'REPORTER': str},
                         nrows=20)
df_k_test.columns = df_k_test.columns.str.lower()

df_v_test = pd.read_csv(vf, sep='\t',
                         dtype={'PRODUCT': str, 'PARTNER': str, 'REPORTER': str},
                         nrows=20)
df_v_test.columns = df_v_test.columns.str.lower()

print(f"\nColumnas kg : {list(df_k_test.columns)}")
print(f"Columnas val: {list(df_v_test.columns)}")

keys_merge = ['reporter', 'partner', 'product', 'flow', 'transport_mode', 'period']
test_merged = pd.merge(df_k_test, df_v_test, on=keys_merge, suffixes=('_kg', '_val'))
print(f"\nMerge de prueba: {len(test_merged)} filas")
print(test_merged[['reporter', 'partner', 'product',
                    'indicator_value_kg', 'indicator_value_val']].head(5).to_string())

del df_k_test, df_v_test, test_merged
gc.collect()


Ficheros _kg.tsv : 4
Ficheros _val.tsv: 4

Columnas kg : ['reporter', 'partner', 'product', 'flow', 'transport_mode', 'period', 'indicators', 'indicator_value']
Columnas val: ['reporter', 'partner', 'product', 'flow', 'transport_mode', 'period', 'indicators', 'indicator_value']

Merge de prueba: 20 filas
  reporter partner product  indicator_value_kg  indicator_value_val
0       AT      AE      08                1000                 1407
1       AT      AE      21                   0                  183
2       AT      AE      25               20800                 1931
3       AT      AE      25               14400                 1766
4       AT      AE      25               12000                  476


0

In [6]:
# =============================================================================
# SESION 4 — Problema detectado: que es ese ':' en indicator_value?
# =============================================================================

df_k_inspect = pd.read_csv(kf, sep='\t',
                             dtype={'PRODUCT': str, 'PARTNER': str, 'REPORTER': str},
                             nrows=2000)
df_k_inspect.columns = df_k_inspect.columns.str.lower()

n_colon   = (df_k_inspect['indicator_value'].astype(str) == ':').sum()
n_numeric = df_k_inspect['indicator_value'].astype(str).str.match(r'^\d').sum()
n_total   = len(df_k_inspect)

print(f"\nDistribucion de indicator_value (muestra 2000 filas):")
print(f"  Valores numericos  : {n_numeric:,}  ({n_numeric/n_total*100:.1f}%)")
print(f"  Valores ':'        : {n_colon:,}  ({n_colon/n_total*100:.1f}%)")
print(f"  Otros              : {n_total - n_numeric - n_colon:,}")

# --- FIGURA 2: proporcion de ':' vs numericos ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

categorias = ['Numerico', 'No disponible (:)', 'Otro']
valores_   = [n_numeric, n_colon, max(0, n_total - n_numeric - n_colon)]
colores_d  = [AZUL_TFM, ROJO_TFM, GRIS_TFM]
wedges, texts, autotexts = axes[0].pie(
    valores_, labels=categorias, colors=colores_d,
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='white'),
    textprops=dict(fontsize=9)
)
for at in autotexts:
    at.set_fontweight('bold')
axes[0].set_title('Composicion de indicator_value\n(muestra 2,000 filas, fichero kg)', pad=12)

no_num = df_k_inspect[
    ~df_k_inspect['indicator_value'].astype(str).str.match(r'^\d', na=False)
]['indicator_value']

top_no_num = no_num.value_counts().head(8).reset_index()
top_no_num.columns = ['valor', 'frecuencia']

axes[1].barh(top_no_num['valor'].astype(str), top_no_num['frecuencia'],
             color=ROJO_TFM, edgecolor='white', height=0.6)

axes[1].set_xlabel('Frecuencia')
axes[1].set_title('Valores no numericos mas frecuentes\n(candidatos a tratar como NaN)')
axes[1].invert_yaxis()

for i, v in enumerate(top_no_num['frecuencia']):
    axes[1].text(v + 0.5, i, str(v), va='center', fontsize=8)

plt.suptitle('Fig. 2 — Calidad del campo indicator_value: problema de codificacion Eurostat',
             fontsize=11, fontweight='bold', y=1.01)

plt.tight_layout()
guardar(fig, 'fig02_calidad_indicator_value.png', fig_path)

del df_k_inspect
gc.collect()


Distribucion de indicator_value (muestra 2000 filas):
  Valores numericos  : 2,000  (100.0%)
  Valores ':'        : 0  (0.0%)
  Otros              : 0
  [figura guardada] fig02_calidad_indicator_value.png


21

In [7]:
# =============================================================================
# SESION 5 — Que hay en 'partner'? Descubro los agregados regionales.
# =============================================================================

df_k_partners = pd.read_csv(kf, sep='\t',
                              dtype={'PRODUCT': str, 'PARTNER': str, 'REPORTER': str})
df_k_partners.columns = df_k_partners.columns.str.lower()

top_partners = df_k_partners['partner'].value_counts().head(40).reset_index()
top_partners.columns = ['partner', 'registros']

mask_agr = top_partners['partner'].str.contains(
    r'EU|WORLD|WRL|EXT_|INT_|_20|EEA|ETH|^W0', na=False, regex=True
)
top_partners['tipo'] = np.where(mask_agr, 'Agregado regional', 'Pais individual')

tabla_resumen(
    top_partners.head(30).assign(
        registros=top_partners.head(30)['registros'].map('{:,}'.format)
    ).set_index('partner'),
    'REVISION — Top 30 codigos en partner (fichero de muestra)',
    col_width=60
)

# Construyo la lista de exclusion a partir de lo que veo
agregados_excluir = [
    'EU27_2020', 'EXT_EU27_2020', 'WORLD', 'WRL_REST', 'EU28', 'EEA',
    'INT_EU27_2020', 'EXT_EA', 'EXT_EA21', 'EXT_EU', 'INT_EA', 'INT_EA21',
    'INT_EU', 'NA', 'ETH', 'EXT', 'W0', 'W00'
]
print(f"\nAgregados identificados para exclusion: {len(agregados_excluir)}")

# --- FIGURA 3: contaminacion de agregados en partner ---
fig, ax = plt.subplots(figsize=(11, 6))
colores_tipo = top_partners['tipo'].map(
    {'Agregado regional': ROJO_TFM, 'Pais individual': AZUL_TFM}
)
bars = ax.barh(top_partners['partner'], top_partners['registros'],
               color=colores_tipo, edgecolor='white', height=0.7)
for bar, val in zip(bars, top_partners['registros']):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', fontsize=7.5)
ax.set_xlabel('Numero de registros')
ax.set_title('Fig. 3 — Top 40 codigos en campo partner: identificacion de agregados regionales\n'
             '(en rojo: no son paises individuales, deben excluirse del analisis)')
ax.invert_yaxis()
legend_handles = [
    mpatches.Patch(color=ROJO_TFM, label='Agregado regional (excluir)'),
    mpatches.Patch(color=AZUL_TFM, label='Pais individual (retener)'),
]
ax.legend(handles=legend_handles, loc='lower right', framealpha=0.9)
plt.tight_layout()
guardar(fig, 'fig03_agregados_partner.png', fig_path)

del df_k_partners
gc.collect()

eu27_codes = [
    'AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'GR', 'ES',
    'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT',
    'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK', 'EL'
]
print(f"Paises EU27 definidos: {len(eu27_codes)}")


------------------------------------------------------------
  REVISION — Top 30 codigos en partner (fichero de muestra)
------------------------------------------------------------
        registros               tipo
partner                             
EXT_EU    104,611  Agregado regional
CN         85,307    Pais individual
US         71,807    Pais individual
IN         55,774    Pais individual
TW         45,054    Pais individual
TR         44,932    Pais individual
TH         43,371    Pais individual
JP         43,067    Pais individual
CA         39,179    Pais individual
KR         38,891    Pais individual
BR         37,902    Pais individual
ID         35,879    Pais individual
IL         32,273    Pais individual
HK         29,897    Pais individual
NO         27,911    Pais individual
VN         27,543    Pais individual
ZA         27,290    Pais individual
MY         26,679    Pais individual
EG         23,861    Pais individual
AU         22,925    Pais individual
MX 

In [8]:
# =============================================================================
# SESION 6 — ETL completo sobre todos los pares
# =============================================================================

print("\n--- ETL iterativo sobre todos los pares kg/val ---")

list_dfs        = []
log_etl         = []
n_brutos_total  = 0
n_validos_total = 0

for kf_path in kg_files:
    vf_path = str(kf_path).replace("_kg.tsv", "_val.tsv")
    if not Path(vf_path).exists():
        print(f"  Sin par val para: {kf_path.name} — omitido")
        continue

    nombre = kf_path.stem.replace("_kg", "")
    print(f"  {nombre} ...", end=" ")

    df_k = pd.read_csv(str(kf_path), sep='\t',
                        dtype={'PRODUCT': str, 'PARTNER': str, 'REPORTER': str})
    df_k.columns = df_k.columns.str.lower()

    df_v = pd.read_csv(vf_path, sep='\t',
                        dtype={'PRODUCT': str, 'PARTNER': str, 'REPORTER': str})
    df_v.columns = df_v.columns.str.lower()

    merged = pd.merge(df_k, df_v, on=keys_merge, suffixes=('_kg', '_val'))
    n_brutos = len(merged)

    merged['kg'] = (
        pd.to_numeric(
            merged['indicator_value_kg'].astype(str).str.replace(':', '', regex=False),
            errors='coerce'
        ).fillna(0).astype('float64')
    )
    merged['eur'] = (
        pd.to_numeric(
            merged['indicator_value_val'].astype(str).str.replace(':', '', regex=False),
            errors='coerce'
        ).fillna(0).astype('float64')
    )

    merged = merged[(merged['eur'] > 0) & (merged['kg'] >= 0)]
    merged = merged[
        merged['reporter'].isin(eu27_codes) &
        (~merged['partner'].isin(agregados_excluir + eu27_codes))
    ]
    merged['hs2_code'] = merged['product'].astype(str).str.zfill(2).str[:2]

    cols_finales = ['reporter', 'partner', 'hs2_code', 'flow',
                    'transport_mode', 'period', 'kg', 'eur']
    n_validos = len(merged)

    log_etl.append({
        'fichero'   : nombre,
        'brutos'    : n_brutos,
        'validos'   : n_validos,
        'retencion' : n_validos / n_brutos * 100 if n_brutos > 0 else 0
    })
    n_brutos_total  += n_brutos
    n_validos_total += n_validos

    print(f"brutos={n_brutos:,}  validos={n_validos:,}  retencion={n_validos/n_brutos*100:.1f}%")

    list_dfs.append(merged[cols_finales])
    del df_k, df_v, merged
    gc.collect()

df_log = pd.DataFrame(log_etl)
tabla_resumen(
    df_log.assign(
        brutos    = df_log['brutos'].map('{:,}'.format),
        validos   = df_log['validos'].map('{:,}'.format),
        retencion = df_log['retencion'].map('{:.1f}%'.format)
    ).set_index('fichero'),
    'TABLA 1 — Resultado del ETL por par de ficheros',
    col_width=72
)

print(f"\nRegistros brutos totales : {n_brutos_total:,}")
print(f"Registros validos totales: {n_validos_total:,}")
print(f"Tasa de retencion global : {n_validos_total/n_brutos_total*100:.1f}%")

# --- FIGURA 4: embudo de retencion por fichero ---
fig, axes = plt.subplots(1, 2, figsize=(13, max(4, len(df_log) * 0.45 + 1.5)))

y_pos = list(range(len(df_log)))
axes[0].barh([y        for y in y_pos], df_log['brutos'],
             color=GRIS_TFM, height=0.4, label='Registros brutos',  align='center')
axes[0].barh([y + 0.42 for y in y_pos], df_log['validos'],
             color=AZUL_TFM, height=0.4, label='Registros validos', align='center')
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(df_log['fichero'], fontsize=8)
axes[0].set_xlabel('Numero de registros')
axes[0].set_title('Brutos vs. Validos por fichero')
axes[0].legend(framealpha=0.9)
axes[0].xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}k')
)

colores_ret = [VERDE_TFM if r >= 50 else NARANJA_TFM if r >= 25 else ROJO_TFM
               for r in df_log['retencion']]
bars = axes[1].barh(df_log['fichero'], df_log['retencion'],
                    color=colores_ret, edgecolor='white', height=0.65)
for bar, r in zip(bars, df_log['retencion']):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                 f'{r:.1f}%', va='center', fontsize=8)
axes[1].axvline(x=df_log['retencion'].mean(), color=GRIS_TFM, linestyle='--',
                linewidth=1.2, label=f"Media: {df_log['retencion'].mean():.1f}%")
axes[1].set_xlabel('Tasa de retencion (%)')
axes[1].set_title('Tasa de retencion por fichero')
axes[1].legend(framealpha=0.9)
axes[1].set_xlim(0, 105)

plt.suptitle('Fig. 4 — Resultado del ETL: embudo de calidad por par de ficheros',
             fontsize=11, fontweight='bold')
plt.tight_layout()
guardar(fig, 'fig04_embudo_etl.png', fig_path)


--- ETL iterativo sobre todos los pares kg/val ---
  import_2002_2007 ... brutos=1,440,721  validos=1,262,245  retencion=87.6%
  import_2008_2013 ... brutos=1,406,561  validos=1,293,819  retencion=92.0%
  import_2014_2019 ... brutos=1,510,080  validos=1,398,053  retencion=92.6%
  import_2020_2025 ... brutos=1,747,095  validos=1,631,983  retencion=93.4%

------------------------------------------------------------------------
  TABLA 1 — Resultado del ETL por par de ficheros
------------------------------------------------------------------------
                     brutos    validos retencion
fichero                                         
import_2002_2007  1,440,721  1,262,245     87.6%
import_2008_2013  1,406,561  1,293,819     92.0%
import_2014_2019  1,510,080  1,398,053     92.6%
import_2020_2025  1,747,095  1,631,983     93.4%
------------------------------------------------------------------------

Registros brutos totales : 6,104,457
Registros validos totales: 5,586,100
Tasa 

In [9]:
# =============================================================================
# SESION 7 — Consolidacion y diagnostico del problema de kg=0
# =============================================================================

df = pd.concat(list_dfs, ignore_index=True)
del list_dfs
gc.collect()

print(f"\nDataset consolidado: {len(df):,} registros | {list(df.columns)}")

n_kg_cero   = (df['kg'] == 0).sum()
print(f"\nRegistros con kg=0: {n_kg_cero:,} ({n_kg_cero/len(df)*100:.2f}%)")

kg_cero_hs2 = (
    df[df['kg'] == 0].groupby('hs2_code').size()
    .reset_index(name='n_kg_cero')
    .sort_values('n_kg_cero', ascending=False)
    .head(20)
)
tabla_resumen(
    kg_cero_hs2.assign(n_kg_cero=kg_cero_hs2['n_kg_cero'].map('{:,}'.format)).set_index('hs2_code'),
    'TABLA 2 — Registros con kg=0 por sector HS2 (top 20)',
)

sanos     = df[df['kg'] > 0]
median_uv = (sanos['eur'] / sanos['kg']).groupby(sanos['hs2_code']).median()

tabla_resumen(
    median_uv.reset_index().rename(columns={0: 'EUR/kg'})
              .sort_values('EUR/kg', ascending=False)
              .assign(**{'EUR/kg': lambda x: x['EUR/kg'].map('{:.2f}'.format)})
              .set_index('hs2_code'),
    'TABLA 3 — Valor unitario mediano por HS2, EUR/kg (base de imputacion de kg=0)',
)

kg_antes = df[df['kg'] > 0]['kg'].copy()

mask_cero = df['kg'] == 0
df.loc[mask_cero, 'kg'] = (
    df.loc[mask_cero, 'eur']
    / df.loc[mask_cero, 'hs2_code'].map(median_uv).fillna(df['eur'].median())
)
print(f"Registros con kg=0 tras imputacion: {(df['kg'] == 0).sum():,}")

# --- FIGURA 5: diagnostico kg=0 y efecto de imputacion ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].barh(kg_cero_hs2['hs2_code'].astype(str), kg_cero_hs2['n_kg_cero'],
             color=ROJO_TFM, edgecolor='white', height=0.7)
axes[0].set_xlabel('Registros con kg = 0')
axes[0].set_title('Registros kg=0\npor codigo HS2 (top 20)')
axes[0].invert_yaxis()

uv_plot = median_uv.reset_index().rename(columns={0: 'uv'}).sort_values('uv', ascending=True)
axes[1].barh(uv_plot['hs2_code'].astype(str), uv_plot['uv'],
             color=AZUL_TFM, edgecolor='white', height=0.7)
axes[1].set_xlabel('EUR / kg (mediana)')
axes[1].set_title('Valor unitario mediano\npor HS2 (base de imputacion)')
axes[1].invert_yaxis()
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

kg_despues  = df[df['kg'] > 0]['kg']
bins_log    = np.logspace(np.log10(0.1), np.log10(kg_antes.quantile(0.99)), 60)
axes[2].hist(kg_antes.clip(upper=kg_antes.quantile(0.99)), bins=bins_log,
             alpha=0.6, color=GRIS_TFM,  label='Antes de imputacion')
axes[2].hist(kg_despues.clip(upper=kg_despues.quantile(0.99)), bins=bins_log,
             alpha=0.5, color=VERDE_TFM, label='Despues de imputacion')
axes[2].set_xscale('log')
axes[2].set_xlabel('Peso (kg, escala log)')
axes[2].set_ylabel('Frecuencia')
axes[2].set_title('Distribucion de kg\nantes vs. despues de imputacion')
axes[2].legend(framealpha=0.9)

plt.suptitle('Fig. 5 — Diagnostico del campo kg: magnitud del problema y efecto de la imputacion',
             fontsize=11, fontweight='bold')
plt.tight_layout()
guardar(fig, 'fig05_diagnostico_kg_imputacion.png', fig_path)


Dataset consolidado: 5,586,100 registros | ['reporter', 'partner', 'hs2_code', 'flow', 'transport_mode', 'period', 'kg', 'eur']

Registros con kg=0: 127,592 (2.28%)

------------------------------------------------------------------------
  TABLA 2 — Registros con kg=0 por sector HS2 (top 20)
------------------------------------------------------------------------
         n_kg_cero
hs2_code          
61           8,469
62           6,676
85           6,103
65           5,606
42           5,471
71           5,161
91           4,952
96           4,676
82           4,516
73           4,354
64           4,268
84           4,164
83           3,816
63           3,479
89           3,201
87           2,650
22           2,572
57           2,536
33           2,492
74           2,452
------------------------------------------------------------------------

------------------------------------------------------------------------
  TABLA 3 — Valor unitario mediano por HS2, EUR/kg (base de imputac

In [10]:
# =============================================================================
# SESION 8 — Campo 'period': conversion temporal
# =============================================================================

print("\nMuestra de valores en 'period':")
print(df['period'].value_counts().head(10).to_string())

df['date'] = pd.to_datetime(df['period'].astype(str), format='%Y%m', errors='coerce')

n_fecha_nula = df['date'].isna().sum()
print(f"\nRegistros con fecha invalida: {n_fecha_nula:,}")
print(f"Rango temporal: {df['date'].min().date()} — {df['date'].max().date()}")


Muestra de valores en 'period':
period
202404    25060
202405    25058
202507    25022
202410    24858
202407    24829
202505    24715
202509    24495
202504    24384
202409    24382
202411    24338

Registros con fecha invalida: 0
Rango temporal: 2002-01-01 — 2025-12-01


In [11]:
# =============================================================================
# SESION 9 — Modos de transporte: inspeccion y filtro maritimo
# =============================================================================

modo_counts = df['transport_mode'].astype(str).value_counts().reset_index()
modo_counts.columns = ['modo', 'registros']

desc_modo = {
    '1': 'Maritimo',           '2': 'Ferroviario',
    '3': 'Carretera',          '4': 'Aereo',
    '5': 'Postal',             '6': 'Instalaciones fijas',
    '7': 'Navegacion interior','9': 'Propulsion propia'
}
modo_counts['descripcion'] = modo_counts['modo'].map(desc_modo).fillna('Desconocido')
modo_counts['pct'] = modo_counts['registros'] / modo_counts['registros'].sum() * 100

tabla_resumen(
    modo_counts.assign(
        registros=modo_counts['registros'].map('{:,}'.format),
        pct      =modo_counts['pct'].map('{:.1f}%'.format)
    ).set_index('modo'),
    'TABLA 4 — Distribucion por modo de transporte (codigos COMEXT)',
)

# --- FIGURA 6: modos de transporte ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colores_modo = [AZUL_TFM if m == '1' else GRIS_TFM for m in modo_counts['modo']]
axes[0].bar(modo_counts['descripcion'], modo_counts['registros'],
            color=colores_modo, edgecolor='white')
axes[0].set_xlabel('Modo de transporte')
axes[0].set_ylabel('Numero de registros')
axes[0].set_title('Registros por modo de transporte\n(azul: maritimo — objeto del TFM)')
axes[0].tick_params(axis='x', rotation=35)
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}k')
)

# Serie temporal de registros maritimos
df_mar_tmp = df[df['transport_mode'].astype(str) == '1'].copy()
df_mar_tmp['date_t'] = pd.to_datetime(df_mar_tmp['period'].astype(str), format='%Y%m', errors='coerce')
mensual = df_mar_tmp.groupby('date_t').size().reset_index(name='n')
axes[1].fill_between(mensual['date_t'], mensual['n'], alpha=0.25, color=AZUL_TFM)
axes[1].plot(mensual['date_t'], mensual['n'], color=AZUL_TFM, linewidth=1.5)
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('Registros maritimos')
axes[1].set_title('Evolucion mensual de registros maritimos\n(cobertura temporal del dataset)')
axes[1].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k')
)

plt.suptitle('Fig. 6 — Modos de transporte: justificacion del filtro maritimo',
             fontsize=11, fontweight='bold')
plt.tight_layout()
guardar(fig, 'fig06_modos_transporte.png', fig_path)

del df_mar_tmp
gc.collect()

df = df[df['transport_mode'].astype(str) == '1'].dropna(subset=['date'])
print(f"\nDataset post-filtro maritimo: {len(df):,} registros")


------------------------------------------------------------------------
  TABLA 4 — Distribucion por modo de transporte (codigos COMEXT)
------------------------------------------------------------------------
      registros descripcion     pct
modo                               
1     5,586,100    Maritimo  100.0%
------------------------------------------------------------------------
  [figura guardada] fig06_modos_transporte.png

Dataset post-filtro maritimo: 5,586,100 registros


In [ ]:
# =============================================================================
# SESION 10 — Datos macroeconomicos: BCE y Brent
# =============================================================================

print("\n--- Inspeccion de datos macro ---")

ecb_raw = pd.read_csv(raw_path / "ecb_rate.csv", parse_dates=['observation_date'])
print(f"BCE — columnas  : {list(ecb_raw.columns)}")
print(f"BCE — registros : {len(ecb_raw)}")
print(ecb_raw.head(5).to_string())

ecb_raw = ecb_raw.rename(columns={'observation_date': 'date', 'ECBMRRFR': 'bce_rate_hist'})
ecb     = ecb_raw.groupby(pd.Grouper(key='date', freq='MS')).mean().reset_index()

brent_raw = pd.read_excel(raw_path / "brent_price.xls", sheet_name="Data 1", skiprows=2)
print(f"\nBrent — columnas: {list(brent_raw.columns)}")
print(brent_raw.head(5).to_string())

brent_raw.columns = ['date', 'oil_price']
brent = brent_raw.groupby(pd.Grouper(key='date', freq='MS')).mean().reset_index()

macro = ecb.merge(brent, on='date', how='outer').ffill().bfill()

tabla_resumen(
    macro.describe().round(4).T,
    'TABLA 5 — Estadisticas descriptivas de las series macro (BCE y Brent)',
)

# --- FIGURA 7: series macro BCE + Brent (doble eje) ---
fig, ax1 = plt.subplots(figsize=(13, 5))

l1, = ax1.plot(macro['date'], macro['bce_rate_hist'],
               color=AZUL_TFM, linewidth=1.8, label='Tipo BCE (%)')
ax1.fill_between(macro['date'], macro['bce_rate_hist'], alpha=0.12, color=AZUL_TFM)
ax1.set_ylabel('Tipo de interes BCE (%)', color=AZUL_TFM)
ax1.tick_params(axis='y', labelcolor=AZUL_TFM)
ax1.set_xlabel('Fecha')

ax2 = ax1.twinx()
l2, = ax2.plot(macro['date'], macro['oil_price'],
               color=NARANJA_TFM, linewidth=1.8, linestyle='--',
               label='Brent (USD/barril)')
ax2.fill_between(macro['date'], macro['oil_price'], alpha=0.08, color=NARANJA_TFM)
ax2.set_ylabel('Precio Brent (USD/barril)', color=NARANJA_TFM)
ax2.tick_params(axis='y', labelcolor=NARANJA_TFM)

eventos = [('2020-03-01', 'COVID-19'), ('2022-02-01', 'Invasion Ucrania'), ('2022-07-01', 'BCE sube tipos')]
for fecha_str, etiqueta in eventos:
    fecha = pd.Timestamp(fecha_str)
    if macro['date'].min() <= fecha <= macro['date'].max():
        ax1.axvline(x=fecha, color=ROJO_TFM, linestyle=':', linewidth=1.2, alpha=0.8)
        ax1.text(fecha, ax1.get_ylim()[1] * 0.88, etiqueta,
                 rotation=90, fontsize=7.5, color=ROJO_TFM, va='top', ha='right')

ax1.legend([l1, l2], [l1.get_label(), l2.get_label()], loc='upper left', framealpha=0.9)
ax1.set_title('Fig. 7 — Series macroeconomicas: tipo BCE y precio del Brent\n'
              '(contexto de costo financiero y energetico del comercio maritimo EU)',
              fontweight='bold')
plt.tight_layout()
guardar(fig, 'fig07_series_macro_bce_brent.png', fig_path)


--- Inspeccion de datos macro ---
BCE — columnas  : ['observation_date', 'ECBMRRFR']
BCE — registros : 9918
  observation_date  ECBMRRFR
0       1999-01-01       3.0
1       1999-01-02       3.0
2       1999-01-03       3.0
3       1999-01-04       3.0
4       1999-01-05       3.0

Brent — columnas: ['Date', 'Europe Brent Spot Price FOB (Dollars per Barrel)']
        Date  Europe Brent Spot Price FOB (Dollars per Barrel)
0 1987-05-15                                             18.58
1 1987-06-15                                             18.86
2 1987-07-15                                             19.86
3 1987-08-15                                             18.98
4 1987-09-15                                             18.31

------------------------------------------------------------------------
  TABLA 5 — Estadisticas descriptivas de las series macro (BCE y Brent)
------------------------------------------------------------------------
               count                    

In [13]:
# =============================================================================
# SESION 11 — Resolucion geografica con pycountry
# =============================================================================

print("\n--- Resolucion geografica ---")

try:
    es_lang     = gettext.translation('iso3166-1', pycountry.LOCALES_DIR, languages=['es'])
    traducir_es = es_lang.gettext
    print("Traduccion castellano cargada")
except Exception as e:
    print(f"Traduccion no disponible: {e}")
    traducir_es = lambda x: x

origin_name_list, o_iso_list = [], []
for iso_code in df['partner']:
    if iso_code in agregados_excluir or iso_code in eu27_codes:
        origin_name_list.append(None); o_iso_list.append(None)
        continue
    fix  = {'EL': 'GR', 'UK': 'GB', 'GR': 'GR'}
    srch = fix.get(str(iso_code).upper(), str(iso_code).upper())
    try:
        c = pycountry.countries.get(alpha_2=srch) or pycountry.countries.get(alpha_3=srch)
        if c:
            origin_name_list.append(traducir_es(c.name)); o_iso_list.append(c.alpha_3)
        else:
            origin_name_list.append(None); o_iso_list.append(None)
    except:
        origin_name_list.append(None); o_iso_list.append(None)

df['origin_name'] = origin_name_list
df['o_iso']       = o_iso_list

n_sin_geo = df['origin_name'].isna().sum()
print(f"\nSocios sin resolucion geografica: {n_sin_geo:,} ({n_sin_geo/len(df)*100:.2f}%)")

no_resueltos = df[df['origin_name'].isna()]['partner'].value_counts().head(15).reset_index()
no_resueltos.columns = ['codigo', 'registros']
tabla_resumen(
    no_resueltos.assign(registros=no_resueltos['registros'].map('{:,}'.format)).set_index('codigo'),
    'TABLA 6 — Codigos de socio sin resolucion geografica (top 15)',
)

df = df.dropna(subset=['origin_name', 'o_iso'])
print(f"Dataset tras limpieza geografica: {len(df):,} registros")

# --- FIGURA 8: top socios por valor importado ---
top_socios = (
    df.groupby('origin_name')['eur'].sum()
    .sort_values(ascending=False).head(25).reset_index()
)
top_socios.columns = ['pais', 'eur_total']

fig, ax = plt.subplots(figsize=(11, 7))
colores_socios = [AZUL_TFM if i < 5 else GRIS_TFM for i in range(len(top_socios))]
ax.barh(top_socios['pais'], top_socios['eur_total'] / 1e9,
        color=colores_socios, edgecolor='white', height=0.7)
ax.set_xlabel('Valor importado acumulado (miles de millones EUR)')
ax.set_title('Fig. 8 — Top 25 socios extracomunitarios por valor de importacion maritima EU\n'
             '(periodo completo del dataset)')
ax.invert_yaxis()
for i, val in enumerate(top_socios['eur_total']):
    ax.text(val / 1e9 + 0.2, i, f'{val/1e9:.1f}B', va='center', fontsize=8)
plt.tight_layout()
guardar(fig, 'fig08_top_socios_valor.png', fig_path)


--- Resolucion geografica ---
Traduccion castellano cargada

Socios sin resolucion geografica: 100,060 (1.79%)

------------------------------------------------------------------------
  TABLA 6 — Codigos de socio sin resolucion geografica (top 15)
------------------------------------------------------------------------
       registros
codigo          
QW        31,246
XS        20,007
QZ        19,343
XC         5,263
XL         4,537
XK         2,714
QP         2,331
AN         2,320
YU         1,245
CS           806
QU           723
XM           256
QS            75
TP            39
QX             6
------------------------------------------------------------------------
Dataset tras limpieza geografica: 5,486,040 registros
  [figura guardada] fig08_top_socios_valor.png


In [14]:
# =============================================================================
# SESION 12 — Distancias CEPII: carga, problema y limpieza
# =============================================================================

print("\n--- Distancias CEPII ---")

dist_df = pd.read_excel(raw_path / "dist_cepii.xls")
print(f"Dimensiones: {dist_df.shape} | Columnas: {list(dist_df.columns)}")
print(f"dtype de 'distw': {dist_df['distw'].dtype}")
print(f"Muestra de valores: {dist_df['distw'].head(10).to_list()}")

n_puntos = (dist_df['distw'].astype(str).str.strip() == '.').sum()
n_nulos  = dist_df['distw'].isna().sum()
print(f"\nValores '.' detectados en distw : {n_puntos:,}")
print(f"Valores NaN detectados en distw : {n_nulos:,}")
print("Decision: pd.to_numeric con errors='coerce' -> rellenar con 8500 km (Head & Mayer 2014)")

dist_df['distw'] = pd.to_numeric(dist_df['distw'], errors='coerce').fillna(8500.0)
print(f"Rango tras limpieza: {dist_df['distw'].min():.0f} — {dist_df['distw'].max():.0f} km")

dist_dict = dist_df.set_index(['iso_o', 'iso_d'])['distw'].to_dict()

iso_map_d = {c.alpha_2: c.alpha_3 for c in pycountry.countries}
iso_map_d.update({'EL': 'GRC', 'UK': 'GBR'})
df['d_iso'] = df['reporter'].map(iso_map_d)

df['dist_nm'] = [
    dist_dict.get((o, d), 8500.0) / 1.852
    for o, d in zip(df['o_iso'], df['d_iso'])
]

n_fallback = (df['dist_nm'] == round(8500.0 / 1.852, 4)).sum()
print(f"Dist_nm min: {df['dist_nm'].min():.0f} MN  max: {df['dist_nm'].max():.0f} MN")
print(f"Pares con distancia por defecto: {n_fallback:,}")

dist_stats_rep = (
    df.groupby('reporter')['dist_nm']
    .agg(['mean', 'median', 'std', 'min', 'max'])
    .round(0).sort_values('mean', ascending=False)
)
tabla_resumen(
    dist_stats_rep.rename(columns={
        'mean': 'Media (MN)', 'median': 'Mediana (MN)',
        'std': 'Desv.Tip.', 'min': 'Min', 'max': 'Max'
    }),
    'TABLA 7 — Estadisticas de distancia (MN) por pais reportante EU27',
)

# --- FIGURA 9: distribucion de distancias ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df['dist_nm'], bins=80, color=AZUL_TFM, edgecolor='white', alpha=0.85)
axes[0].axvline(df['dist_nm'].mean(), color=ROJO_TFM, linestyle='--', linewidth=1.5,
                label=f"Media: {df['dist_nm'].mean():.0f} MN")
axes[0].axvline(df['dist_nm'].median(), color=NARANJA_TFM, linestyle=':', linewidth=1.5,
                label=f"Mediana: {df['dist_nm'].median():.0f} MN")
axes[0].set_xlabel('Distancia ortodromica (Millas Nauticas)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribucion de distancias\npor registro de flujo maritimo')
axes[0].legend(framealpha=0.9)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

dist_por_socio = (
    df.groupby('origin_name')['dist_nm'].mean()
    .sort_values(ascending=False).head(20).reset_index()
)
axes[1].barh(dist_por_socio['origin_name'], dist_por_socio['dist_nm'],
             color=AZUL_TFM, edgecolor='white', height=0.7)
axes[1].set_xlabel('Distancia media (Millas Nauticas)')
axes[1].set_title('Top 20 socios por distancia media\n(rutas mas largas al mercado EU)')
axes[1].invert_yaxis()
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.suptitle('Fig. 9 — Distancias CEPII: distribucion y ranking de rutas maritimas',
             fontsize=11, fontweight='bold')
plt.tight_layout()
guardar(fig, 'fig09_distribucion_distancias.png', fig_path)


--- Distancias CEPII ---
Dimensiones: (50176, 14) | Columnas: ['iso_o', 'iso_d', 'contig', 'comlang_off', 'comlang_ethno', 'colony', 'comcol', 'curcol', 'col45', 'smctry', 'dist', 'distcap', 'distw', 'distwces']
dtype de 'distw': object
Muestra de valores: [25.09354, 13168.22, 9587.316, 976.8974, 9091.576, 7570.084, 239.9064, 12773.08, 5187.788, 11106.96]

Valores '.' detectados en distw : 2,215
Valores NaN detectados en distw : 0
Decision: pd.to_numeric con errors='coerce' -> rellenar con 8500 km (Head & Mayer 2014)
Rango tras limpieza: 1 — 19781 km
Dist_nm min: 122 MN  max: 10550 MN
Pares con distancia por defecto: 0

------------------------------------------------------------------------
  TABLA 7 — Estadisticas de distancia (MN) por pais reportante EU27
------------------------------------------------------------------------
          Media (MN)  Mediana (MN)  Desv.Tip.     Min      Max
reporter                                                      
RO            4590.0        459

In [15]:
# =============================================================================
# SESION 13 — Merge con macro y variable hs2_int
# =============================================================================

df = df.merge(macro, on='date', how='left').ffill().bfill()
df['hs2_int'] = pd.to_numeric(df['hs2_code']).astype('int16')

print(f"Registros sin bce_rate_hist: {df['bce_rate_hist'].isna().sum():,}")
print(f"Registros sin oil_price    : {df['oil_price'].isna().sum():,}")
print(f"Dataset completo: {len(df):,} registros")

Registros sin bce_rate_hist: 0
Registros sin oil_price    : 0
Dataset completo: 5,486,040 registros


In [16]:
# =============================================================================
# SESION 14 — Clasificacion sectorial HS2
# =============================================================================

print("\n--- Mapeo HS2 a Macro-Sector ---")
print(df['hs2_code'].value_counts().head(20).to_string())

hs2_sector_map = {}
for hs_raw in df['hs2_code'].unique():
    try:
        h = int(hs_raw)
        if   1  <= h <= 24:         hs2_sector_map[hs_raw] = 'Alimentos & Agricultura'
        elif h  == 27:              hs2_sector_map[hs_raw] = 'Energia & Combustibles'
        elif h  == 30:              hs2_sector_map[hs_raw] = 'Farmacia & Salud'
        elif 28 <= h <= 38:         hs2_sector_map[hs_raw] = 'Quimica'
        elif 39 <= h <= 40:         hs2_sector_map[hs_raw] = 'Plasticos & Caucho'
        elif 50 <= h <= 64:         hs2_sector_map[hs_raw] = 'Textil, Ropa & Calzado'
        elif h in [84, 85, 90]:     hs2_sector_map[hs_raw] = 'Tecnologia & Electronica'
        elif h in [86, 87, 88, 89]: hs2_sector_map[hs_raw] = 'Movilidad & Automocion'
        elif 72 <= h <= 83:         hs2_sector_map[hs_raw] = 'Metales & Minerales'
        else:                       hs2_sector_map[hs_raw] = 'Otros Sectores'
    except:
        hs2_sector_map[hs_raw] = 'Otros Sectores'

df['macro_sector'] = df['hs2_code'].map(hs2_sector_map).fillna('Otros Sectores')

sector_stats = (
    df.groupby('macro_sector')
    .agg(n_registros=('eur','count'), eur_total=('eur','sum'),
         kg_total=('kg','sum'), dist_media=('dist_nm','mean'))
    .sort_values('eur_total', ascending=False)
)
sector_stats['pct_valor'] = sector_stats['eur_total'] / sector_stats['eur_total'].sum() * 100

tabla_resumen(
    sector_stats.assign(
        n_registros=sector_stats['n_registros'].map('{:,}'.format),
        eur_total  =sector_stats['eur_total'].map('{:,.0f}'.format),
        kg_total   =sector_stats['kg_total'].map('{:,.0f}'.format),
        dist_media =sector_stats['dist_media'].map('{:,.0f} MN'.format),
        pct_valor  =sector_stats['pct_valor'].map('{:.1f}%'.format)
    ),
    'TABLA 8 — Estadisticas por Macro-Sector (dataset maritimo filtrado)',
    col_width=90
)

# --- FIGURA 10: estructura sectorial ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sec_sorted = sector_stats.sort_values('eur_total', ascending=True)
paleta_sec = plt.cm.Blues(np.linspace(0.35, 0.85, len(sec_sorted)))

axes[0].barh(sec_sorted.index, sec_sorted['eur_total'] / 1e9,
             color=paleta_sec, edgecolor='white', height=0.7)
axes[0].set_xlabel('Valor importado acumulado (B EUR)')
axes[0].set_title('Valor total por sector')

axes[1].barh(sec_sorted.index, sec_sorted['n_registros'] / 1e3,
             color=paleta_sec, edgecolor='white', height=0.7)
axes[1].set_xlabel('Numero de registros (miles)')
axes[1].set_title('Numero de registros por sector')

axes[2].barh(sec_sorted.index, sec_sorted['dist_media'],
             color=paleta_sec, edgecolor='white', height=0.7)
axes[2].set_xlabel('Distancia media (MN)')
axes[2].set_title('Distancia media de ruta por sector')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.suptitle('Fig. 10 — Estructura sectorial: valor, volumen y distancia por Macro-Sector',
             fontsize=11, fontweight='bold')
plt.tight_layout()
guardar(fig, 'fig10_estructura_sectorial.png', fig_path)


--- Mapeo HS2 a Macro-Sector ---
hs2_code
84    227057
85    198105
73    178509
87    168659
09    162324
61    154622
08    151275
62    143932
22    143274
63    140332
76    133566
72    124621
03    123936
68    115110
83    114929
21    112165
12    111437
82    109546
33    108905
96    108884

------------------------------------------------------------------------------------------
  TABLA 8 — Estadisticas por Macro-Sector (dataset maritimo filtrado)
------------------------------------------------------------------------------------------
                         n_registros          eur_total            kg_total dist_media pct_valor
macro_sector                                                                                    
Energia & Combustibles       105,354  5,747,003,003,820  15,286,116,839,065   3,171 MN     35.3%
Tecnologia & Electronica     425,162  3,172,057,934,855     338,965,606,406   3,915 MN     19.5%
Metales & Minerales          873,412  1,479,618,188,384 

In [17]:
# =============================================================================
# SESION 15 — Entrenamiento ML: alpha resiliencia nodal
# =============================================================================

print("\n--- Modelo ML: alpha resiliencia nodal ---")

n_sample = min(len(df), 300_000)
df_reg   = df.sample(n=n_sample, random_state=42)

features_ml = ['dist_nm', 'hs2_int', 'oil_price']
target_ml   = df_reg['eur'] * 0.012

model = HistGradientBoostingRegressor(max_iter=40)
model.fit(df_reg[features_ml], target_ml)

df['alpha_resiliencia_nodal'] = model.predict(df[features_ml]).astype('float32')

y_pred_train = model.predict(df_reg[features_ml])
y_true_train = target_ml.values
residuos     = y_pred_train - y_true_train
r2_manual    = 1 - np.sum(residuos**2) / np.sum((y_true_train - y_true_train.mean())**2)
mae_manual   = np.abs(residuos).mean()

tabla_resumen(
    pd.DataFrame({
        'Metrica': ['R2 (muestra entreno)', 'MAE (muestra entreno)', 'Registros', 'Features'],
        'Valor'  : [f'{r2_manual:.4f}', f'{mae_manual:.2f} EUR', f'{n_sample:,}', str(features_ml)]
    }).set_index('Metrica'),
    'TABLA 9 — Diagnostico del modelo ML (HistGradientBoostingRegressor)',
)

# --- FIGURA 11: diagnostico modelo ML ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

idx_plot = np.random.choice(len(y_true_train), size=min(5000, len(y_true_train)), replace=False)
axes[0].scatter(y_true_train[idx_plot], y_pred_train[idx_plot],
                alpha=0.25, s=8, color=AZUL_TFM)
lim = max(y_true_train[idx_plot].max(), y_pred_train[idx_plot].max())
axes[0].plot([0, lim], [0, lim], color=ROJO_TFM, linewidth=1.2, linestyle='--')
axes[0].set_xlabel('Valor real (EUR * 0.012)')
axes[0].set_ylabel('Prediccion modelo')
axes[0].set_title(f'Real vs. Predicho\n(R2 = {r2_manual:.3f})')

axes[1].hist(
    np.clip(residuos, np.percentile(residuos, 1), np.percentile(residuos, 99)),
    bins=60, color=AZUL_TFM, edgecolor='white', alpha=0.85
)
axes[1].axvline(0, color=ROJO_TFM, linestyle='--', linewidth=1.2)
axes[1].set_xlabel('Residuo (pred - real)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title(f'Distribucion de residuos\n(MAE = {mae_manual:.2f} EUR)')

alpha_sector = df.groupby('macro_sector')['alpha_resiliencia_nodal'].mean().sort_values(ascending=True)
axes[2].barh(alpha_sector.index, alpha_sector.values,
             color=AZUL_TFM, edgecolor='white', height=0.7)
axes[2].set_xlabel('Alpha resiliencia nodal (media)')
axes[2].set_title('Alpha medio por Macro-Sector')

plt.suptitle('Fig. 11 — Diagnostico del modelo ML: alpha de resiliencia nodal',
             fontsize=11, fontweight='bold')
plt.tight_layout()
guardar(fig, 'fig11_diagnostico_ml.png', fig_path)

del df_reg
gc.collect()


--- Modelo ML: alpha resiliencia nodal ---

------------------------------------------------------------------------
  TABLA 9 — Diagnostico del modelo ML (HistGradientBoostingRegressor)
------------------------------------------------------------------------
                                                     Valor
Metrica                                                   
R2 (muestra entreno)                                0.4262
MAE (muestra entreno)                         44609.69 EUR
Registros                                          300,000
Features               ['dist_nm', 'hs2_int', 'oil_price']
------------------------------------------------------------------------
  [figura guardada] fig11_diagnostico_ml.png


23394

In [18]:
# =============================================================================
# SESION 16 — Calculo de LVaR_95
# =============================================================================

print("\n--- Calculo LVaR_95 ---")

df['lt_std_base'] = (
    df.groupby(['d_iso', 'hs2_int'])['dist_nm']
    .transform('std')
    .fillna(500) / 336
)

df['LVaR_95'] = (
    1.645 * df['lt_std_base'] * df['eur'] * (df['bce_rate_hist'] / 100 / 365)
).astype('float32')

print(f"lt_std_base media    : {df['lt_std_base'].mean():.4f} dias")
print(f"LVaR_95 media        : {df['LVaR_95'].mean():.6f} EUR")
print(f"LVaR_95 percentil 95 : {df['LVaR_95'].quantile(0.95):.6f} EUR")

lvar_sector = df.groupby('macro_sector')['LVaR_95'].agg(
    ['mean', 'median', 'std', lambda x: x.quantile(0.95)]
)
lvar_sector.columns = ['Media', 'Mediana', 'Desv.Tip.', 'P95']

tabla_resumen(
    lvar_sector.sort_values('Media', ascending=False).round(6),
    'TABLA 10 — LVaR_95 (EUR) por Macro-Sector: estadisticos clave',
)

print("\nDiagnostico de estabilidad numerica:")

# Correccion: Cortar los valores negativos generados por el modelo ML a 0
df['alpha_resiliencia_nodal'] = df['alpha_resiliencia_nodal'].clip(lower=0)

for col in ['alpha_resiliencia_nodal', 'LVaR_95']:
    n_neg = (df[col] < 0).sum()
    n_inf = np.isinf(df[col]).sum()
    n_nan = df[col].isna().sum()
    status = "OK" if n_neg == 0 and n_inf == 0 and n_nan == 0 else "REVISAR"
    print(f"  {col:35} neg={n_neg:,}  inf={n_inf:,}  nan={n_nan:,}  [{status}]")

# --- FIGURA 12: LVaR_95 por sector y evolucion temporal ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

lvar_s    = df.groupby('macro_sector')['LVaR_95'].mean().sort_values(ascending=True)
paleta_lv = plt.cm.Reds(np.linspace(0.35, 0.85, len(lvar_s)))
axes[0].barh(lvar_s.index, lvar_s.values, color=paleta_lv, edgecolor='white', height=0.7)
axes[0].set_xlabel('LVaR_95 medio (EUR)')
axes[0].set_title('LVaR_95 medio por Macro-Sector\n(riesgo de variabilidad en lead time)')

lvar_mensual = df.groupby('date')['LVaR_95'].mean().reset_index()
axes[1].fill_between(lvar_mensual['date'], lvar_mensual['LVaR_95'], alpha=0.2, color=ROJO_TFM)
axes[1].plot(lvar_mensual['date'], lvar_mensual['LVaR_95'], color=ROJO_TFM, linewidth=1.6)
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('LVaR_95 medio (EUR)')
axes[1].set_title('Evolucion temporal del LVaR_95 medio\n(efecto de tipos BCE y valor del flujo)')

plt.suptitle('Fig. 12 — LVaR_95: riesgo logistico financiero por sector y periodo',
             fontsize=11, fontweight='bold')
plt.tight_layout()
guardar(fig, 'fig12_lvar95_sector_tiempo.png', fig_path)


--- Calculo LVaR_95 ---
lt_std_base media    : 5.2857 dias
LVaR_95 media        : 1248.374023 EUR
LVaR_95 percentil 95 : 2423.370898 EUR

------------------------------------------------------------------------
  TABLA 10 — LVaR_95 (EUR) por Macro-Sector: estadisticos clave
------------------------------------------------------------------------
                                 Media    Mediana     Desv.Tip.            P95
macro_sector                                                                  
Energia & Combustibles    22436.687500  84.283241  75799.324886  122897.846875
Tecnologia & Electronica   3288.682617   7.522688  31480.937399    9192.368994
Movilidad & Automocion     2925.624023   8.282656  22367.076356    7695.420947
Metales & Minerales         730.254822   4.165650   5222.859451    2435.376746
Quimica                     580.611633   4.703214   4504.214174    1970.717310
Alimentos & Agricultura     467.214020   7.468991   3117.220567    1557.509888
Textil, Ropa & Calz

In [19]:
# =============================================================================
# SESION 17 — Inspeccion del fichero de puertos
# =============================================================================

print("\n--- Fichero de puertos ---")

ports = pd.read_csv(raw_path / "Ports.csv")
print(f"Registros: {len(ports)} | Columnas: {list(ports.columns)}")
print(ports.head(10).to_string())

ports['share'] = pd.to_numeric(ports['share_country_maritime_import'], errors='coerce').fillna(0)

top_ports = (
    ports[ports['share'] > 0][['ISO3', 'portname', 'share']]
    .sort_values('share', ascending=False).head(20)
)
tabla_resumen(
    top_ports.assign(share=top_ports['share'].map('{:.1f}%'.format)).set_index('portname'),
    'TABLA 11 — Top 20 puertos por cuota de trafico maritimo nacional',
)


--- Fichero de puertos ---
Registros: 2033 | Columnas: ['X', 'Y', 'portid', 'portname', 'country', 'ISO3', 'continent', 'fullname', 'lat', 'lon', 'vessel_count_total', 'vessel_count_container', 'vessel_count_dry_bulk', 'vessel_count_general_cargo', 'vessel_count_RoRo', 'vessel_count_tanker', 'industry_top1', 'industry_top2', 'industry_top3', 'share_country_maritime_import', 'share_country_maritime_export', 'LOCODE', 'pageid', 'countrynoaccents', 'ObjectId']
              X             Y    portid      portname         country ISO3       continent                   fullname        lat         lon  vessel_count_total  vessel_count_container  vessel_count_dry_bulk  vessel_count_general_cargo  vessel_count_RoRo  vessel_count_tanker         industry_top1                    industry_top2                    industry_top3  share_country_maritime_import  share_country_maritime_export  LOCODE                            pageid countrynoaccents  ObjectId
0  1.514731e+07  4.254924e+06  port1325   

In [20]:
# =============================================================================
# SESION 18 — Parametros arancelarios
# =============================================================================

socios_pref = [
    'MAR', 'TUR', 'GBR', 'NOR', 'CHE', 'EGY', 'TUN',
    'DZA', 'ISR', 'JOR', 'UKR', 'SRB', 'ISL', 'ALB', 'MKD'
]

friccion_sec = {
    'Alimentos & Agricultura'  : 0.085,
    'Textil, Ropa & Calzado'   : 0.12,
    'Tecnologia & Electronica' : 0.025,
    'Farmacia & Salud'         : 0.015,
    'Metales & Minerales'      : 0.005,
    'Movilidad & Automocion'   : 0.045,
    'Energia & Combustibles'   : 0.0,
    'Quimica'                  : 0.03,
    'Plasticos & Caucho'       : 0.065
}

tabla_resumen(
    pd.DataFrame([
        {'Sector': k, 'Arancel base': f'{v:.1%}',
         'Con TLC (x0.1)': f'{v*0.1:.2%}',
         'Ruta larga (+50%)': f'{v*1.5:.2%}'}
        for k, v in sorted(friccion_sec.items(), key=lambda x: x[1], reverse=True)
    ]).set_index('Sector'),
    'TABLA 12 — Parametros arancelarios: base, TLC y penalizacion por distancia',
    col_width=82
)


----------------------------------------------------------------------------------
  TABLA 12 — Parametros arancelarios: base, TLC y penalizacion por distancia
----------------------------------------------------------------------------------
                         Arancel base Con TLC (x0.1) Ruta larga (+50%)
Sector                                                                
Textil, Ropa & Calzado          12.0%          1.20%            18.00%
Alimentos & Agricultura          8.5%          0.85%            12.75%
Plasticos & Caucho               6.5%          0.65%             9.75%
Movilidad & Automocion           4.5%          0.45%             6.75%
Quimica                          3.0%          0.30%             4.50%
Tecnologia & Electronica         2.5%          0.25%             3.75%
Farmacia & Salud                 1.5%          0.15%             2.25%
Metales & Minerales              0.5%          0.05%             0.75%
Energia & Combustibles           0.0%         

In [21]:
# =============================================================================
# SESION 19 — Exportacion final al Parquet
# =============================================================================

PATH_PARQUET = str(PATH_BASE / "outputs" / "simulation_inputs" / "TFM_HS2_Macro_Dataset.parquet")
Path(PATH_PARQUET).parent.mkdir(parents=True, exist_ok=True)

print(f"\n--- Exportacion final ---")
print(f"Destino: {PATH_PARQUET}")

if os.path.exists(PATH_PARQUET):
    print("Fichero previo encontrado — eliminando para regenerar limpio")
    os.remove(PATH_PARQUET)

writer            = None
n_paises          = 0
n_puertos         = 0
n_registros_total = 0

for d_iso, group in df.groupby('d_iso'):
    country_ports = ports[(ports['ISO3'] == d_iso) & (ports['share'] > 0)]
    if country_ports.empty:
        continue

    n_paises += 1

    for _, port in country_ports.iterrows():
        chunk = group.copy()
        w     = port['share'] / 100

        chunk['puerto']       = port['portname']
        dist_efectiva         = chunk['dist_nm'] * 1.65
        chunk['lead_time']    = (dist_efectiva / 336) + 8.5
        chunk['lt_std']       = chunk['lead_time'] * 0.18

        for col in ['eur', 'kg', 'LVaR_95', 'alpha_resiliencia_nodal']:
            chunk[col] *= w

        chunk['arancel_base']     = chunk['macro_sector'].map(friccion_sec).fillna(0.045)
        chunk['factor_tlc']       = np.where(chunk['o_iso'].isin(socios_pref), 0.1, 1.0)
        chunk['factor_distancia'] = np.where(chunk['dist_nm'] > 2500, 1.5, 1.0)
        chunk['tariff_rate']      = chunk['arancel_base'] * chunk['factor_tlc'] * chunk['factor_distancia']
        chunk['costo_arancel']    = chunk['eur'] * chunk['tariff_rate']

        chunk['costo_co2_ets'] = (
            dist_efectiva * 1.852
            * (chunk['kg'] / 1000)
            * 0.000045
            * 85
        )

        chunk['total_friction_cost'] = (
            chunk['alpha_resiliencia_nodal']
            + chunk['costo_co2_ets']
            + chunk['costo_arancel']
        )

        table = pa.Table.from_pandas(chunk.reset_index(drop=True))
        if writer is None:
            writer = pq.ParquetWriter(PATH_PARQUET, table.schema, compression='snappy')
        writer.write_table(table)

        n_puertos         += 1
        n_registros_total += len(chunk)

        del chunk
        gc.collect()

if writer:
    writer.close()

print(f"Exportacion completada.")
print(f"  Paises exportados  : {n_paises}")
print(f"  Puertos escritos   : {n_puertos}")
print(f"  Registros totales  : {n_registros_total:,}")
print(f"  Tamano del archivo : {Path(PATH_PARQUET).stat().st_size / 1e6:.1f} MB")


--- Exportacion final ---
Destino: /Users/hugoa/Library/CloudStorage/OneDrive-UniversitatdeBarcelona/UB/Tesina/TFM_SIT2026/outputs/simulation_inputs/TFM_HS2_Macro_Dataset.parquet
Fichero previo encontrado — eliminando para regenerar limpio
Exportacion completada.
  Paises exportados  : 22
  Puertos escritos   : 297
  Registros totales  : 93,043,658
  Tamano del archivo : 4081.2 MB


In [ ]:
# =============================================================================
# SESION 20 — Verificacion exhaustiva del Parquet generado
# =============================================================================

print("\n--- Verificacion post-exportacion ---")

df_check = pd.read_parquet(PATH_PARQUET).head(5000)

vars_check = ['eur', 'kg', 'dist_nm', 'lead_time', 'LVaR_95',
              'alpha_resiliencia_nodal', 'total_friction_cost',
              'tariff_rate', 'costo_co2_ets', 'costo_arancel']

tabla_resumen(
    df_check[vars_check].describe().round(6),
    'TABLA 13 — Estadisticas de las variables clave del Parquet (muestra 5,000 filas)',
    col_width=100
)

# Verificacion 1
n_friction_neg = (df_check['total_friction_cost'] < 0).sum()
print(f"\nVERIFICACION 1 — total_friction_cost negativos     : {n_friction_neg}  "
      f"[{'OK' if n_friction_neg == 0 else 'ERROR'}]")

# Verificacion 2
componentes = (df_check['alpha_resiliencia_nodal']
               + df_check['costo_co2_ets']
               + df_check['costo_arancel'])
diff_max = (componentes - df_check['total_friction_cost']).abs().max()
print(f"VERIFICACION 2 — Max diferencia suma componentes vs total: {diff_max:.8f}  "
      f"[{'OK' if diff_max < 1e-4 else 'ERROR'}]")

# Verificacion 3
print("VERIFICACION 3 — Infinitos y NaN por variable:")
for col in vars_check:
    n_inf = np.isinf(df_check[col]).sum()
    n_nan = df_check[col].isna().sum()
    status = "OK" if n_inf == 0 and n_nan == 0 else "REVISAR"
    print(f"  {col:35} inf={n_inf}  nan={n_nan}  [{status}]")

print(f"\nVERIFICACION 4 — Macro-sectores presentes:")
print(df_check['macro_sector'].value_counts().to_string())

# --- FIGURA 13: descomposicion del total_friction_cost ---
friction_desglose = df_check.groupby('macro_sector').agg(
    alpha   = ('alpha_resiliencia_nodal', 'mean'),
    co2     = ('costo_co2_ets',           'mean'),
    arancel = ('costo_arancel',            'mean'),
).sort_values('alpha', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

bottom         = np.zeros(len(friction_desglose))
colores_comp   = [AZUL_TFM, VERDE_TFM, NARANJA_TFM]
labels_comp    = ['Alpha resiliencia (ML)', 'costo CO2 ETS', 'Costo arancelario']

for col, color, label in zip(['alpha', 'co2', 'arancel'], colores_comp, labels_comp):
    axes[0].barh(friction_desglose.index, friction_desglose[col],
                 left=bottom, color=color, edgecolor='white', height=0.7, label=label)
    bottom += friction_desglose[col].values

axes[0].set_xlabel('Costo medio (EUR)')
axes[0].set_title('Descomposicion del costo de friccion total\npor Macro-Sector')
axes[0].legend(loc='lower right', framealpha=0.9)
axes[0].invert_yaxis()

total_comp = friction_desglose[['alpha', 'co2', 'arancel']].sum()
wedges, texts, autotexts = axes[1].pie(
    total_comp, labels=['Alpha resiliencia', 'CO2 ETS', 'Arancel'],
    colors=colores_comp, autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='white'),
)
for at in autotexts:
    at.set_fontweight('bold')
axes[1].set_title('Proporcion de cada componente\nsobre total_friction_cost')

plt.suptitle('Fig. 13 — Descomposicion del costo total de friccion logistica (verificacion matematica)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
guardar(fig, 'fig13_descomposicion_friction_cost.png', fig_path)

# --- FIGURA 14: panel de validacion final ---
fig = plt.figure(figsize=(16, 9))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

ax1 = fig.add_subplot(gs[0, 0])
lt_sector = df_check.groupby('macro_sector')['lead_time'].mean().sort_values(ascending=True)
ax1.barh(lt_sector.index, lt_sector.values, color=AZUL_TFM, edgecolor='white', height=0.7)
ax1.set_xlabel('Lead time medio (dias)')
ax1.set_title('Lead time medio por sector')

ax2 = fig.add_subplot(gs[0, 1])
sample_val = df_check.sample(min(1500, len(df_check)), random_state=1)
ax2.scatter(sample_val['dist_nm'], sample_val['lead_time'],
            alpha=0.25, s=8, color=AZUL_TFM)
ax2.set_xlabel('Distancia (MN)')
ax2.set_ylabel('Lead time (dias)')
ax2.set_title('Correlacion distancia — lead time')
corr_val = sample_val[['dist_nm', 'lead_time']].corr().iloc[0, 1]
ax2.text(0.05, 0.92, f'r = {corr_val:.3f}', transform=ax2.transAxes,
         fontsize=9, color=ROJO_TFM)

ax3 = fig.add_subplot(gs[0, 2])
tr_sector = df_check.groupby('macro_sector')['tariff_rate'].mean().sort_values(ascending=True)
ax3.barh(tr_sector.index, tr_sector.values * 100, color=NARANJA_TFM, edgecolor='white', height=0.7)
ax3.set_xlabel('Tipo arancelario efectivo medio (%)')
ax3.set_title('Tariff rate efectiva por sector')

ax4 = fig.add_subplot(gs[1, 0])
ax4.scatter(sample_val['eur'], sample_val['LVaR_95'],
            alpha=0.25, s=8, color=ROJO_TFM)
ax4.set_xlabel('Valor del flujo (EUR)')
ax4.set_ylabel('LVaR_95 (EUR)')
ax4.set_title('LVaR_95 en funcion del valor del flujo')
ax4.set_xscale('log')
ax4.set_yscale('log')

ax5 = fig.add_subplot(gs[1, 1])
ax5.hist(df_check['lead_time'], bins=50, color=AZUL_TFM, edgecolor='white', alpha=0.85)
ax5.axvline(df_check['lead_time'].mean(), color=ROJO_TFM, linestyle='--', linewidth=1.3,
            label=f"Media: {df_check['lead_time'].mean():.1f} dias")
ax5.set_xlabel('Lead time (dias)')
ax5.set_ylabel('Frecuencia')
ax5.set_title('Distribucion del lead time')
ax5.legend(framealpha=0.9)

ax6 = fig.add_subplot(gs[1, 2])
fc_socio = df_check.groupby('origin_name')['total_friction_cost'].mean().sort_values(ascending=False).head(12)
ax6.barh(fc_socio.index, fc_socio.values, color=GRIS_TFM, edgecolor='white', height=0.7)
ax6.set_xlabel('Friction cost medio (EUR)')
ax6.set_title('Top 12 socios por friction cost medio')
ax6.invert_yaxis()

fig.suptitle('Fig. 14 — Panel de validacion final: coherencia de las variables clave del dataset',
             fontsize=12, fontweight='bold', y=1.01)
guardar(fig, 'fig14_panel_validacion_final.png', fig_path)


# =============================================================================
# Resumen ejecutivo del proceso
# =============================================================================
print("\n" + "=" * 78)
print("  RESUMEN EJECUTIVO DEL PROCESO")
print("=" * 78)
resumen_final = pd.DataFrame([
    ['Pares de ficheros procesados',       f'{len(kg_files)}'],
    ['Registros brutos totales',           f'{n_brutos_total:,}'],
    ['Registros validos post-ETL',         f'{n_validos_total:,}'],
    ['Tasa de retencion global',           f'{n_validos_total/n_brutos_total*100:.1f}%'],
    ['Filtro aplicado',                    'transport_mode = 1 (maritimo)'],
    ['Rango temporal cubierto',            f"{df['date'].min().date()} — {df['date'].max().date()}"],
    ['Socios extracomunitarios unicos',    f"{df['o_iso'].nunique()}"],
    ['Paises importadores EU27 activos',   f"{df['reporter'].nunique()}"],
    ['Macro-sectores HS2',                 f"{df['macro_sector'].nunique()}"],
    ['Agregados regionales excluidos',     f'{len(agregados_excluir)}'],
    ['Registros kg imputados',             f'{mask_cero.sum():,}'],
    ['Paises con puertos asignados',       f'{n_paises}'],
    ['Puertos de entrada modelizados',     f'{n_puertos}'],
    ['Registros en Parquet final',         f'{n_registros_total:,}'],
    ['Tamano Parquet (MB)',                f'{Path(PATH_PARQUET).stat().st_size / 1e6:.1f}'],
    ['Figuras generadas',                  '14'],
    ['Tablas de revision generadas',       '13'],
]).set_index(0)
resumen_final.index.name = 'Indicador'
resumen_final.columns    = ['Valor']
print(resumen_final.to_string())
print("=" * 78)
print(f"Figuras:  {fig_path}")
print(f"Parquet:  {PATH_PARQUET}")
print("=" * 78)


--- Verificacion post-exportacion ---

----------------------------------------------------------------------------------------------------
  TABLA 13 — Estadisticas de las variables clave del Parquet (muestra 5,000 filas)
----------------------------------------------------------------------------------------------------
                eur            kg      dist_nm    lead_time       LVaR_95  alpha_resiliencia_nodal  total_friction_cost  tariff_rate  costo_co2_ets  costo_arancel
count  5.000000e+03  5.000000e+03  5000.000000  5000.000000   5000.000000              5000.000000         5.000000e+03  5000.000000   5.000000e+03   5.000000e+03
mean   1.174899e+06  4.745803e+06  5750.574116    36.739426   1225.814819             17652.509766         5.371086e+05     0.083913   4.367940e+05   8.266212e+04
std    3.928386e+06  2.719071e+07  2464.770671    12.103785   4255.970215             34712.644531         2.809028e+06     0.059974   2.753895e+06   2.630513e+05
min    2.509200e+00  3.